# Chapter 4 — Build It Yourself: Attention

The hands-on heart of [Chapter 4](../README.md). You'll build attention's engine — a
weighted average of the past — then a real self-attention head, then train it on Shakespeare
and generate text. **✍️ Your turn** cells (hint + "Stuck? Show the answer"), **📖 Study & run**
cells, **▶️ Check your work** cells. Top to bottom. 🚀

## Step 1 — Setup, data, batches  ▶️ Run this

Load ~1MB of Shakespeare, build the character vocabulary, and define `get_batch`. Tensors
here are 3-D: **`(B, T, C)`** = (Batch, Time/positions, Channels/embedding-dims).

In [ ]:
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F

DATA = next(p for p in [Path("../data/input.txt"), Path("data/input.txt"),
                        Path("chapters/04-attention/data/input.txt")] if p.exists())
text = DATA.read_text()
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join(itos[i] for i in ids)

torch.manual_seed(1337)
block_size, n_embd, batch_size = 32, 64, 32
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]

def get_batch(split):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i:i + block_size] for i in ix])
    y = torch.stack([d[i + 1:i + block_size + 1] for i in ix])
    return x, y

print(f"{len(text)} chars | vocab {vocab_size} | block_size {block_size}")
xb, yb = get_batch("train")
print("a batch:", tuple(xb.shape), "->", tuple(yb.shape), "  (B, T)")

## Step 2 — Attention's engine: a weighted average of the past  ✍️ Your turn

Build a lower-triangular, row-normalized matrix `wei` (the uniform "average of the past"
weights), then aggregate `x` with it. `wei @ x` gives every position the average of `x` up to
and including itself.

<details><summary>Stuck? Show the answer</summary>

```python
wei = tril / tril.sum(1, keepdim=True)   # normalize each row to sum to 1
xbow = wei @ x                           # (T, T) @ (B, T, C) → (B, T, C)
```
</details>

In [ ]:
torch.manual_seed(1337)
B, T, C = 4, 8, 2
x = torch.randn(B, T, C)
tril = torch.tril(torch.ones(T, T))

# ✍️ make wei lower-triangular with each row summing to 1, then aggregate the past
wei = tril        # replace: normalize each row of tril to sum to 1
xbow = x          # replace: the weighted average of the past = wei applied to x
print("wei row 2:", [round(v, 2) for v in wei[2].tolist()])   # expect [0.33, 0.33, 0.33, 0, 0, 0, 0, 0]
print("xbow shape:", tuple(xbow.shape))

In [ ]:
# ▶️ Check your work
try:
    assert abs(wei[2, 0].item() - 1/3) < 1e-4 and wei[2, 3].item() == 0.0, "row 2 should be [1/3, 1/3, 1/3, 0, ...]"
    assert abs(wei[3].sum().item() - 1.0) < 1e-5, "each row of wei should sum to 1"
    assert tuple(xbow.shape) == (B, T, C), f"xbow should be {(B, T, C)}, got {tuple(xbow.shape)}"
    assert torch.allclose(xbow[0, 0], x[0, 0]), "position 0's 'average of the past' is just itself"
    assert torch.allclose(xbow[0, 2], x[0, :3].mean(0), atol=1e-5), "xbow[0,2] should be the average of x over positions 0,1,2 — apply wei to x with @"
    print("✅ The weighted-average trick works — a triangular matmul averages the past. This IS attention's engine.")
except (NameError, AssertionError) as e:
    print("❌", e)

## Step 3 — A real self-attention head  ✍️ Your turn (the heart of the chapter)

Now make the weights *smart*. Each token emits a **query**, **key**, and **value**. The
score from token *i* to *j* is query·key; we mask the future, softmax into weights, and take
the weighted sum of values. Fill the **two** marked lines:
1. the **scaled** attention scores (query · key, times `head_size ** -0.5`);
2. the head **output** (weights applied to the values).

<details><summary>Stuck? Show the answer</summary>

```python
wei = q @ k.transpose(-2, -1) * head_size ** -0.5    # #1
...
out = wei @ v                                          # #2
```
</details>

In [ ]:
torch.manual_seed(1337)
B, T, C, head_size = 4, 8, 32, 16
x = torch.randn(B, T, C)
key   = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)
k, q, v = key(x), query(x), value(x)
tril = torch.tril(torch.ones(T, T))

# ✍️ #1: attention scores — every query dotted with every key, then scaled by 1/sqrt(head_size)
wei = q @ k.transpose(-2, -1)                     # ← scale these scores down so they don't blow up with head_size
wei = wei.masked_fill(tril == 0, float("-inf"))   # mask the future (given)
wei = F.softmax(wei, dim=-1)                        # → weights that sum to 1 (given)

# ✍️ #2: the head output — the weighted sum of the values
out = v                                           # ← replace: combine the weights wei with values v

print("wei", tuple(wei.shape), "| out", tuple(out.shape))

In [ ]:
# ▶️ Check your work
try:
    assert tuple(wei.shape) == (B, T, T), f"wei should be (B,T,T)=({B},{T},{T}), got {tuple(wei.shape)}"
    assert tuple(out.shape) == (B, T, head_size), f"out should be (B,T,{head_size}), got {tuple(out.shape)}"
    assert wei[0, 0, 1].item() == 0.0, "causal mask: token 0 must NOT attend to the future (wei[0,0,1] = 0)"
    assert abs(wei[0, 3].sum().item() - 1.0) < 1e-5, "each row of attention weights should sum to 1"
    assert not torch.allclose(out[0, 3], v[0, 3]), "out should be a weighted SUM of values (wei @ v), not v itself"
    _unscaled = F.softmax((q @ k.transpose(-2, -1)).masked_fill(tril == 0, float("-inf")), dim=-1)
    assert not torch.allclose(wei, _unscaled, atol=1e-6), "the scores aren't scaled — multiply by head_size ** -0.5 (the 1/sqrt scaling)"
    print("✅ Attention head works — causal, scaled, weights sum to 1, and the output blends the past values.")
except (NameError, AssertionError, TypeError, AttributeError) as e:
    print("❌", e)

## Step 4 — Package it: the model  📖 Study & run

The `Head` class is exactly your Step 3 code, tidied into an `nn.Module`. The model adds a
**token** embedding (what each char is) + a **position** embedding (where it sits), runs the
head, and projects to `vocab_size` logits.

In [ ]:
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
    def forward(self, x):
        B, T, C = x.shape
        k, q = self.key(x), self.query(x)
        wei = q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1)
        return wei @ self.value(x)

class AttentionLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, n_embd)
        self.position_embedding = nn.Embedding(block_size, n_embd)
        self.sa_head = Head(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)
    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.token_embedding(idx) + self.position_embedding(torch.arange(T))
        logits = self.lm_head(self.sa_head(x))
        loss = None
        if targets is not None:
            B, T, Vc = logits.shape
            loss = F.cross_entropy(logits.view(B * T, Vc), targets.view(B * T))
        return logits, loss

model = AttentionLM()
print(sum(p.nelement() for p in model.parameters()), "parameters")

## Step 5 — Train it  📖 Study & run

Same reset → backward → update loop, with the AdamW optimizer (a better one — Chapter 7).
~3000 steps, a few seconds. Watch the loss fall below the bigram's ~2.5.

In [ ]:
@torch.no_grad()
def estimate_loss(eval_iters=40):
    out = {}
    model.eval()
    for split in ["train", "val"]:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            _, loss = model(*get_batch(split))
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
for step_i in range(3000):
    _, loss = model(*get_batch("train"))
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if step_i % 1000 == 0:
        L = estimate_loss()
        print(f"step {step_i:4d} | train {L['train']:.4f} | val {L['val']:.4f}")

val_loss = estimate_loss()["val"]
print(f"final val loss {val_loss:.4f}  (bigram baseline ~2.5)")

In [ ]:
# ▶️ Check your work
try:
    assert val_loss < 2.45, f"val loss should beat the bigram's ~2.5, got {val_loss:.4f}"
    print(f"✅ Trained! val loss {val_loss:.4f} — below the bigram baseline. One attention head is learning.")
except (NameError, AssertionError) as e:
    print("❌", e)

## Step 6 — Generate Shakespeare(ish)  ✍️ Your turn

The autoregressive loop: take the last `block_size` chars, get the next-char probabilities,
**sample** one, and **append** it. Fill the two marked lines.

<details><summary>Stuck? Show the answer</summary>

```python
idx_next = torch.multinomial(probs, num_samples=1)   # sample one token
idx = torch.cat([idx, idx_next], dim=1)              # append it
```
</details>

In [ ]:
@torch.no_grad()
def generate(max_new_tokens):
    idx = torch.zeros((1, 1), dtype=torch.long)        # start from char 0 (a newline)
    for _ in range(max_new_tokens):
        logits, _ = model(idx[:, -block_size:])        # condition on the last block_size chars
        probs = F.softmax(logits[:, -1, :], dim=-1)    # next-char distribution
        # ✍️ sample one token from probs, then append it to idx
        idx_next = probs[:, :1]      # replace: sample one token from probs (multinomial)
        idx = idx                    # replace: append idx_next onto idx (cat along dim=1)
    return idx

print(decode(generate(300)[0].tolist()))

In [ ]:
# ▶️ Check your work
try:
    out_text = decode(generate(200)[0].tolist())
    assert len(out_text) >= 200, "should generate ~200 characters — did you append each new token?"
    assert all(c in chars for c in out_text), "output should be valid characters"
    print(f"✅ It generates! Structured gibberish that's learning the shape of English:\n\n{out_text[:160]}")
except (NameError, AssertionError, RuntimeError) as e:
    print("❌", e)

## 🎓 You built attention.

A weighted average of the past, with *learned* weights from query·key — masked to be causal,
positioned so order matters. One head already learns the shape of English. **Stacking many
heads and layers is the Transformer — [Chapter 5](../../05-transformer/).**

Next: the [exercises](../exercises/) (visualize the attention weights, break the mask) and
the [mini-project](../project/), *The Attention Microscope*. 👋